# Rank August's best rows

*Data Analysis in Python · Topic 2.4*

The weekly meeting wants August 2024's five biggest revenue rows. The paid social team wants its top 5 rows by sessions, and it doesn't want a row left out just because it's tied with another one.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the file into the `export` table.
2. In the first cell marked ✏️, create the `august` table: the rows of the `export` table dated in August 2024. The file also has two days of September.
3. In the second cell marked ✏️, create the `top_revenue` table: the five rows of the `august` table with the highest revenue, highest first, with the columns `date`, `session_source`, `session_medium`, `session_campaign` and `revenue`.
4. In the third cell marked ✏️, create:
   - `paid_social`: the `august` table's rows whose `session_medium` is `paid_social`, as a copy
   - a `sessions_rank` column in the `paid_social` table: each row's place by sessions, 1 for the most sessions, with tied rows sharing the best place they tie for
5. In the last cell marked ✏️, create:
   - `tied_fifth`: the number of rows in fifth place
   - `top_paid_social`: the rows of the `paid_social` table in the top 5 places, ties included
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `august` table has 1,712 rows. The `top_revenue` table has five rows, with revenue from $2,371.88 down to $1,666.76. The `paid_social` table has 574 rows and a `sessions_rank` column, `tied_fifth` is `6`, and the `top_paid_social` table has 10 rows.

## Parameters

In [ ]:
import pandas as pd

FILE_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_08.csv"

In [ ]:
export = pd.read_csv(FILE_URL, parse_dates=["date"])
export.shape

## August only

In [ ]:
# ✏️ Create august: the rows dated in August 2024


## Top revenue rows

In [ ]:
# ✏️ Create top_revenue


## Paid social, ranked

In [ ]:
# ✏️ Create paid_social and its sessions_rank column


In [ ]:
# ✏️ Create tied_fifth and top_paid_social


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def frame(name):
        value = found.get(name)
        return value if isinstance(value, pd.DataFrame) else None

    august = frame("august")
    if august is None:
        results.append((False, "august isn't a DataFrame yet. Filter export to the rows dated on or before \"2024-08-31\"."))
    elif len(august) == 1712:
        results.append((True, "august has its 1,712 rows, without September's."))
    elif len(august) == 1820:
        results.append((False, "august has all 1,820 rows of the file, including September 1 and 2. Keep the rows with export[\"date\"] <= \"2024-08-31\"."))
    else:
        results.append((False, f"august has {len(august)} rows, but it should have 1,712: every row dated in August."))

    expected_revenue = [2371.88, 1911.67, 1819.70, 1745.90, 1666.76]
    top = frame("top_revenue")
    columns = ["date", "session_source", "session_medium", "session_campaign", "revenue"]
    if top is None:
        results.append((False, "top_revenue isn't a DataFrame yet. Sort august by revenue, highest first, and take the first five rows."))
    elif "revenue" not in top.columns:
        results.append((False, f"top_revenue has no revenue column. Its columns should be {columns}."))
    else:
        revenue = [round(float(value), 2) for value in top["revenue"]]
        if revenue == expected_revenue and top.columns.tolist() == columns:
            results.append((True, "top_revenue has August's five biggest revenue rows, highest first."))
        elif revenue == expected_revenue:
            results.append((False, f"top_revenue has the right rows, but its columns are {top.columns.tolist()}. Select {columns}."))
        elif 1679.90 in revenue:
            results.append((False, "top_revenue has a row from September 1 ($1,679.90). Sort august, not export."))
        elif sorted(revenue, reverse=True) == expected_revenue:
            results.append((False, "top_revenue has the right rows, but not highest first. Sort with ascending=False."))
        elif revenue and min(revenue) < 0:
            results.append((False, "top_revenue has the smallest revenue rows: refunds. Sort with ascending=False to put the highest first."))
        else:
            results.append((False, f"top_revenue's revenue should be {expected_revenue}, but it is {revenue}."))

    paid = frame("paid_social")
    if paid is None:
        results.append((False, "paid_social isn't a DataFrame yet. Filter august to session_medium == \"paid_social\", and add .copy()."))
    elif len(paid) != 574:
        results.append((False, f"paid_social has {len(paid)} rows, but August has 574 paid social rows. Filter august, not export."))
    elif "sessions_rank" not in paid.columns:
        results.append((False, "paid_social has no sessions_rank column. Add it with paid_social[\"sessions_rank\"] = paid_social[\"sessions\"].rank(...)."))
    else:
        top_rank = paid.loc[paid["sessions"].idxmax(), "sessions_rank"]
        ranks_at_81 = set(paid.loc[paid["sessions"] == 81, "sessions_rank"])
        if top_rank != 1:
            results.append((False, f"The row with the most sessions has sessions_rank {top_rank}, not 1. Rank with ascending=False."))
        elif ranks_at_81 == {5}:
            results.append((True, "paid_social's sessions_rank puts tied rows in the best place they share."))
        elif ranks_at_81 == {7.5}:
            results.append((False, "Six rows tie with 81 sessions, and they're ranked 7.5: the default tie rule averages their places. Use method=\"min\", so they share 5th place."))
        elif len(ranks_at_81) > 1:
            results.append((False, f"The six rows with 81 sessions have different ranks, {sorted(ranks_at_81)}: method=\"first\" breaks ties by row order. Use method=\"min\", so they share 5th place."))
        else:
            results.append((False, f"The six rows with 81 sessions should share 5th place, but their rank is {sorted(ranks_at_81)}. Use rank(ascending=False, method=\"min\")."))

    tied = found.get("tied_fifth")
    top_paid = frame("top_paid_social")
    if isinstance(tied, (pd.DataFrame, pd.Series)):
        results.append((False, f"tied_fifth is a {type(tied).__name__}, not a number. Count the rows instead of keeping them: (paid_social[\"sessions_rank\"] == 5).sum()."))
    elif tied == 6:
        results.append((True, "Six rows share fifth place."))
    elif tied == 0:
        results.append((False, "tied_fifth is 0: no row is ranked exactly 5. Check the tie rule in sessions_rank: with method=\"min\", six rows share 5th place."))
    else:
        results.append((False, f"tied_fifth should be the number of rows whose sessions_rank is 5: (paid_social[\"sessions_rank\"] == 5).sum(). It is {tied!r}."))
    if top_paid is not None and len(top_paid) == 10:
        results.append((True, "top_paid_social has 10 rows: the top 5 places, with the tie."))
    elif top_paid is not None and len(top_paid) == 5:
        results.append((False, "top_paid_social has 5 rows, so it cut the tie in fifth place. Keep the rows with sessions_rank <= 5 instead of taking the first five."))
    else:
        results.append((False, "top_paid_social should be the rows of paid_social with sessions_rank <= 5: 10 rows, because six rows share fifth place."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. A head(5) would have left out 5 of the 6 tied rows.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
august = export[export["date"] <= "2024-08-31"]
august.shape
```

```python
top_revenue = august.sort_values("revenue", ascending=False).head(5)
top_revenue = top_revenue.loc[:, ["date", "session_source", "session_medium", "session_campaign", "revenue"]]
top_revenue
```

```python
paid_social = august[august["session_medium"] == "paid_social"].copy()
paid_social["sessions_rank"] = paid_social["sessions"].rank(ascending=False, method="min")
paid_social.sort_values("sessions_rank").head(12)
```

```python
tied_fifth = (paid_social["sessions_rank"] == 5).sum()
top_paid_social = paid_social[paid_social["sessions_rank"] <= 5]
tied_fifth, len(top_paid_social)
```

</details>